# U2-03 · Del perceptrón a TensorFlow: entrenar, validar y probar

*Programación de Redes Neuronales - Machine Learning (IFCD48) · Unidad 2, apartados 2.1, 2.3, 2.4 y 2.5*

Ejecuta cada celda con **Mayús + Intro** (en Colab, también con el botón ▶). Sin GPU funciona igual; el apartado 3 tarda unos minutos.

> Datos: MNIST, de Y. LeCun, C. Cortes y C. J. C. Burges (CC BY-SA 3.0), y Fashion-MNIST, de Zalando Research (licencia MIT). Keras los descarga en la primera ejecución.

## 1. El perceptrón de Rosenblatt con MNIST

In [ ]:
import numpy as np
import keras
import matplotlib.pyplot as plt

(x_ent, y_ent), (x_pru, y_pru) = keras.datasets.mnist.load_data()

def perceptron(X, y, epocas=50):
    """Regla de Rosenblatt; y vale -1 o +1. Devuelve pesos y
    errores por época."""
    w, b, errores = np.zeros(X.shape[1]), 0.0, []
    for _ in range(epocas):
        fallos = 0
        for xi, yi in zip(X, y):
            if yi * (xi @ w + b) <= 0:      # mal clasificado
                w += yi * xi                # corrige hacia la clase
                b += yi
                fallos += 1
        errores.append(fallos)
        if fallos == 0:                     # separa todo: termina
            break
    return w, b, errores

for a, c in [(0, 1), (3, 8)]:
    sel, selp = np.isin(y_ent, [a, c]), np.isin(y_pru, [a, c])
    X = x_ent[sel].reshape(-1, 784) / 255
    y = np.where(y_ent[sel] == c, 1, -1)
    w, b, err = perceptron(X, y)
    Xp = x_pru[selp].reshape(-1, 784) / 255
    yp = np.where(y_pru[selp] == c, 1, -1)
    acierto = np.mean(np.sign(Xp @ w + b) == yp)
    print(f"{a} frente a {c}: {len(y)} imágenes, {len(err)} épocas, "
          f"fallos {err[:3]}...{err[-3:]}, prueba {acierto:.4f}")
    plt.plot(range(1, len(err) + 1), err, "o-", ms=2,
             label=f"{a} frente a {c}")
plt.yscale("symlog")
plt.xlabel("época"); plt.ylabel("errores en entrenamiento")
plt.legend(); plt.show()  # plt.savefig("perceptron.pdf")

## 2. La cinta de gradientes frente a la retropropagación a mano

Las dos primeras celdas guardan como archivos el código de los apartados 2.2 y 2.3, que la tercera importa.

In [ ]:
%%writefile l_espiral.py
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPClassifier

def espiral(n=300, ruido=0.25, semilla=0):
    """Dos brazos de espiral entrelazados: clases 0 y 1."""
    rng = np.random.default_rng(semilla)
    t = np.sqrt(rng.uniform(0, 1, n)) * 3 * np.pi
    clase = rng.integers(0, 2, n)
    r = t * np.where(clase == 1, 1, -1)
    X = np.column_stack([r * np.cos(t), r * np.sin(t)]) / 3
    return X + rng.normal(0, ruido, X.shape), clase

if __name__ == "__main__":       # solo al ejecutar este archivo
    X, y = espiral()
    Xp, yp = espiral(semilla=1)            # otra muestra para probar
    xx, yy = np.meshgrid(np.linspace(-3.5, 3.5, 200),
                         np.linspace(-3.5, 3.5, 200))
    fig, ejes = plt.subplots(1, 3, figsize=(4.4, 1.6))
    for eje, n in zip(ejes, [2, 8, 64]):
        red = MLPClassifier((n,), activation="tanh", solver="lbfgs",
                            max_iter=5000,
                            random_state=0).fit(X, y)
        zona = red.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1]
        eje.contourf(xx, yy, zona.reshape(xx.shape),
                     levels=[0, .5, 1], colors=["#CFE0F2", "#F6DDBB"])
        eje.scatter(*X.T, c=np.where(y, "#E08A1E", "#1F5FA8"), s=2)
        eje.set_title(f"{n} neuronas", loc="center")
        eje.set_xticks([]); eje.set_yticks([])
        print(f"{n:2d} neuronas ocultas: entrenamiento "
              f"{red.score(X, y):.3f}, "
              f"prueba {red.score(Xp, yp):.3f}")
    fig.savefig("espiral.pdf")

In [ ]:
%%writefile m_retro.py
import numpy as np
from l_espiral import espiral              # datos del apartado 2.2

X, y = espiral()
y = y.reshape(-1, 1).astype(float)

def iniciar(semilla=0):
    rng = np.random.default_rng(semilla)
    return {"W1": rng.normal(0, 1, (2, 16)), "b1": np.zeros(16),
            "W2": rng.normal(0, 0.25, (16, 1)), "b2": np.zeros(1)}

def adelante(p, X):
    H = np.tanh(X @ p["W1"] + p["b1"])            # capa oculta
    P = 1 / (1 + np.exp(-(H @ p["W2"] + p["b2"])))  # salida
    return H, P

def perdida(p, X=X, y=y):
    _, P = adelante(p, X)
    return -np.mean(y * np.log(P) + (1 - y) * np.log(1 - P))

def atras(p, X=X, y=y):
    """Retropropagación: la regla de la cadena, de la salida
    hacia la entrada."""
    H, P = adelante(p, X)
    dZ2 = (P - y) / len(X)                 # error en la salida
    dH = dZ2 @ p["W2"].T                   # se reparte hacia atrás
    dZ1 = dH * (1 - H ** 2)                # derivada de tanh
    return {"W2": H.T @ dZ2, "b2": dZ2.sum(0),
            "W1": X.T @ dZ1, "b1": dZ1.sum(0)}

if __name__ == "__main__":
    p = iniciar()
    # comprobación: derivada numérica de tres pesos al azar
    g, h = atras(p), 1e-6
    for nombre, idx in [("W1", (0, 3)), ("W1", (1, 7)),
                        ("W2", (5, 0))]:
        p[nombre][idx] += h; f1 = perdida(p)
        p[nombre][idx] -= 2 * h; f2 = perdida(p)
        p[nombre][idx] += h
        print(f"{nombre}{idx}: retropropagación "
              f"{g[nombre][idx]:+.6f}, "
              f"numérica {(f1 - f2) / (2 * h):+.6f}")

    # entrenamiento: descenso de gradiente con estos gradientes
    for paso in range(1, 20_001):
        for k, v in atras(p).items():
            p[k] -= 0.5 * v
        if paso in (1, 100, 1000, 20_000):
            acierto = np.mean((adelante(p, X)[1] > 0.5) == y)
            print(f"paso {paso:5d}: pérdida {perdida(p):.4f}, "
                  f"acierto {acierto:.3f}")

In [ ]:
import numpy as np
import tensorflow as tf
from m_retro import X, y, iniciar, atras

p = iniciar()                       # los mismos pesos iniciales
v = {k: tf.Variable(a) for k, a in p.items()}
with tf.GradientTape() as cinta:    # TensorFlow anota cada paso
    H = tf.tanh(X @ v["W1"] + v["b1"])
    logit = H @ v["W2"] + v["b2"]
    perdida = tf.reduce_mean(tf.nn.sigmoid_cross_entropy_with_logits(
        labels=y, logits=logit))
g_tf = cinta.gradient(perdida, v)           # y deriva hacia atrás

g_np = atras(p)
for k in p:
    dif = np.max(np.abs(g_tf[k].numpy() - g_np[k]))
    print(f"{k}: diferencia máxima {dif:.1e}")

## 3. Fashion-MNIST y cinco decisiones de entrenamiento

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import keras

(x, y), _ = keras.datasets.fashion_mnist.load_data()
clases = ["camiseta", "pantalón", "jersey", "vestido", "abrigo",
          "sandalia", "camisa", "zapatilla", "bolso", "botín"]
fig, ejes = plt.subplots(1, 10, figsize=(4.4, 0.75))
for c, eje in enumerate(ejes):
    eje.imshow(x[np.flatnonzero(y == c)[0]])
    eje.set_title(clases[c], fontsize=5, loc="center",
                  fontweight="normal")
    eje.axis("off")
plt.show()  # fig.savefig("fashion.pdf")
print(x.shape, np.bincount(y))

In [ ]:
import time
import numpy as np
import keras

(x, y), (x_pru, y_pru) = keras.datasets.fashion_mnist.load_data()
x, x_pru = x / 255.0, x_pru / 255.0
x_ent, y_ent = x[:50_000], y[:50_000]       # entrenamiento
x_val, y_val = x[50_000:], y[50_000:]       # validación

def entrenar(optimizador, epocas=10, lote=32, inicial="he_normal"):
    keras.utils.set_random_seed(0)
    red = keras.Sequential([
        keras.Input(shape=(28, 28)), keras.layers.Flatten(),
        keras.layers.Dense(256, activation="relu",
                           kernel_initializer=inicial),
        keras.layers.Dense(128, activation="relu",
                           kernel_initializer=inicial),
        keras.layers.Dense(10, activation="softmax")])
    red.compile(optimizador, "sparse_categorical_crossentropy",
                metrics=["accuracy"])
    t0 = time.perf_counter()
    h = red.fit(x_ent, y_ent, epochs=epocas, batch_size=lote,
                validation_data=(x_val, y_val), verbose=0)
    return red, h.history, time.perf_counter() - t0

opt = keras.optimizers
print("1) tasa de aprendizaje (SGD, 10 épocas)")
for eta in [0.001, 0.01, 0.1, 1.0]:
    _, h, _ = entrenar(opt.SGD(eta))
    print(f"   eta = {eta:<5}: "
          f"validación {h['val_accuracy'][-1]:.4f}")

print("2) optimizador (10 épocas)")
curvas = {}
for nombre, o in [("SGD", opt.SGD(0.01)),
                  ("SGD + momento", opt.SGD(0.01, momentum=0.9)),
                  ("Adam", opt.Adam(0.001))]:
    _, h, s = entrenar(o)
    curvas[nombre] = h["val_accuracy"]
    print(f"   {nombre:<14}: validación {h['val_accuracy'][-1]:.4f}"
          f" ({s:.0f} s)")

print("3) tamaño de lote (Adam, 10 épocas)")
for lote in [16, 128, 1024]:
    _, h, s = entrenar(opt.Adam(0.001), lote=lote)
    print(f"   lote {lote:4d}: validación "
          f"{h['val_accuracy'][-1]:.4f}, {s:.0f} s")

print("4) pesos iniciales (Adam, 5 épocas)")
for inicial in ["zeros", "he_normal"]:
    _, h, _ = entrenar(opt.Adam(0.001), epocas=5, inicial=inicial)
    print(f"   {inicial:<9}: validación {h['val_accuracy'][-1]:.4f}")
np.savez("curvas_opt.npz", **curvas)

**Prueba.** Repite la comparación de optimizadores con 20 épocas. ¿Sigue sin ayudar el momento?

## 4. Parada temprana y evaluación final

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import keras

(x, y), (x_pru, y_pru) = keras.datasets.fashion_mnist.load_data()
x, x_pru = x / 255.0, x_pru / 255.0
clases = ["camiseta", "pantalón", "jersey", "vestido", "abrigo",
          "sandalia", "camisa", "zapatilla", "bolso", "botín"]

keras.utils.set_random_seed(0)
red = keras.Sequential([
    keras.Input(shape=(28, 28)), keras.layers.Flatten(),
    keras.layers.Dense(256, activation="relu"),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dense(10, activation="softmax")])
red.compile("adam", "sparse_categorical_crossentropy",
            metrics=["accuracy"])
parada = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True)
h = red.fit(x[:50_000], y[:50_000], epochs=60,
            validation_data=(x[50_000:], y[50_000:]),
            callbacks=[parada], verbose=0)
mejor = int(np.argmin(h.history["val_loss"]))
print(f"se detiene en la época {len(h.history['loss'])}; "
      f"la mejor fue la {mejor + 1}")
print(f"pérdida en la mejor época: entrenamiento "
      f"{h.history['loss'][mejor]:.3f}, "
      f"validación {h.history['val_loss'][mejor]:.3f}")

# una sola evaluación final, en prueba
perdida, acierto = red.evaluate(x_pru, y_pru, verbose=0)
print(f"prueba: acierto {acierto:.4f}")
pred = red.predict(x_pru, verbose=0).argmax(1)
for c in range(10):
    print(f"  {clases[c]:<10} {np.mean(pred[y_pru == c] == c):.3f}")

epocas = np.arange(1, len(h.history["loss"]) + 1)
plt.plot(epocas, h.history["loss"], label="entrenamiento")
plt.plot(epocas, h.history["val_loss"], label="validación")
plt.axvline(mejor + 1, color="#9AA5B1", ls="--")
plt.xlabel("época"); plt.ylabel("pérdida (entropía cruzada)")
plt.legend(); plt.show()  # plt.savefig("parada.pdf")

## 5. Tres maneras de escribir la red y guardarla

In [ ]:
import numpy as np
import tensorflow as tf
import keras

(x, y), (x_pru, y_pru) = keras.datasets.fashion_mnist.load_data()
x, x_pru = x.astype("float32") / 255, x_pru.astype("float32") / 255

# 1) tensores y variables
t = tf.constant([[1.0, 2.0], [3.0, 4.0]])
w = tf.Variable(tf.ones((2, 1)))
print("tensor", t.shape, t.dtype,
      "| t @ w =", (t @ w).numpy().ravel())

# 2) la misma red, de tres formas
def secuencial():
    return keras.Sequential([
        keras.Input(shape=(28, 28)), keras.layers.Flatten(),
        keras.layers.Dense(128, activation="relu"),
        keras.layers.Dense(10)])

def funcional():
    entrada = keras.Input(shape=(28, 28))
    h = keras.layers.Flatten()(entrada)
    h = keras.layers.Dense(128, activation="relu")(h)
    return keras.Model(entrada, keras.layers.Dense(10)(h))

class Subclase(keras.Model):
    def __init__(self):
        super().__init__()
        self.aplanar = keras.layers.Flatten()
        self.oculta = keras.layers.Dense(128, activation="relu")
        self.salida = keras.layers.Dense(10)
    def call(self, entrada):
        return self.salida(self.oculta(self.aplanar(entrada)))

perdida = keras.losses.SparseCategoricalCrossentropy(from_logits=True)
for nombre, crear in [("Sequential", secuencial),
                      ("funcional", funcional),
                      ("subclase", Subclase)]:
    keras.utils.set_random_seed(0)
    red = crear()
    red.compile("adam", perdida, metrics=["accuracy"])
    red.fit(x, y, epochs=3, verbose=0)
    print(f"{nombre:<10}: {red.count_params():,} parámetros, prueba "
          f"{red.evaluate(x_pru, y_pru, verbose=0)[1]:.4f}")

# 3) el mismo entrenamiento con un bucle propio
keras.utils.set_random_seed(0)
red, optim = secuencial(), keras.optimizers.Adam()
datos = tf.data.Dataset.from_tensor_slices((x, y)).shuffle(
    60_000, seed=0).batch(32)
for epoca in range(3):
    for xb, yb in datos:
        with tf.GradientTape() as cinta:
            p = perdida(yb, red(xb, training=True))
        grads = cinta.gradient(p, red.trainable_variables)
        optim.apply_gradients(zip(grads, red.trainable_variables))
acierto = np.mean(red.predict(x_pru, verbose=0).argmax(1) == y_pru)
print(f"bucle propio: prueba {acierto:.4f}")

# 4) guardar, cargar y comprobar
red.save("ropa.keras")
cargada = keras.models.load_model("ropa.keras")
iguales = np.allclose(red.predict(x_pru[:100], verbose=0),
                      cargada.predict(x_pru[:100], verbose=0))
print("el modelo cargado predice lo mismo:", iguales)

**Ejercicio.** Añade una tercera capa oculta de 64 neuronas a la versión funcional y entrénala con parada temprana. ¿Mejora la camisa?

> Xiao, H., Rasul, K. y Vollgraf, R. (2017). Fashion-MNIST: a novel image dataset for benchmarking machine learning algorithms. arXiv:1708.07747. https://doi.org/10.48550/arXiv.1708.07747
>
> Chollet, F. (2021). *Deep Learning with Python* (2.ª ed.). Manning. ISBN 978-1-61729-686-4